In [0]:
# Preparar notebook
from pyspark.sql import functions as F
from pyspark.sql import types as T
from datetime import datetime

print("Testing & Data Quality Framework")
print("Spark version:", spark.version)

In [0]:
# Creando función de Validación
def run_quality_test( df, table_name, test_name, condition, description):
    """
    Ejecuta una prueba de calidad sobre un DataFrame.

    condition:
        Expresión PySpark que identifica registros inválidos.
    """

    total_records = df.count()

    failed_records = (
        df.filter(condition).count()
    )

    status = "PASS" if failed_records == 0 else "FAIL"

    return {
        "table_name": table_name,
        "test_name": test_name,
        "description": description,
        "total_records": total_records,
        "failed_records": failed_records,
        "status": status,
        "execution_timestamp": datetime.utcnow()
    }

In [0]:
# Primera prueba Campos Obligatorios
df_customers = spark.table(
    "banco.banking_analytics.silver_customers"
)

result_customer_id = run_quality_test(
    df=df_customers,
    table_name="silver_customers",
    test_name="CUSTOMER_ID_NOT_NULL",
    condition=F.col("customer_id").isNull(),
    description="Customer ID must not be null"
)

display([result_customer_id])

In [0]:
# Segunda Prueba Unicidad de Customer_ID
duplicate_customers = (
    df_customers
    .groupBy("customer_id")
    .count()
    .filter(F.col("count") > 1)
)

result_customer_unique = run_quality_test(
    df=duplicate_customers,
    table_name="silver_customers",
    test_name="CUSTOMER_ID_UNIQUE",
    condition=F.col("count") > 1,
    description="Customer ID must be unique"
)

display([result_customer_unique])

In [0]:
# Tercera Prueba: Formato del correo electronico
invalid_email_condition = (
    F.col("email").isNotNull() &
    ~F.col("email").rlike(
        r"^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$"
    )
)

result_email = run_quality_test(
    df=df_customers,
    table_name="silver_customers",
    test_name="EMAIL_FORMAT_VALID",
    condition=invalid_email_condition,
    description="Non-null emails must have a valid format"
)

display([result_email])

In [0]:
# Prueba de Integridad Referencial
df_accounts = spark.table(
    "banco.banking_analytics.silver_accounts"
)

invalid_customer_references = (
    df_accounts.alias("a")
    .join(
        df_customers.select("customer_id").alias("c"),
        F.col("a.customer_id") == F.col("c.customer_id"),
        "left_anti"
    )
)

result_account_customer_fk = run_quality_test(
    df=invalid_customer_references,
    table_name="silver_accounts",
    test_name="CUSTOMER_REFERENCE_VALID",
    condition=F.lit(True),
    description="Every account must reference an existing customer"
)

display([result_account_customer_fk])

In [0]:
# Pueba de Reglas del Negocio: Saldo Negativo
result_account_balance = run_quality_test(
    df=df_accounts,
    table_name="silver_accounts",
    test_name="ACCOUNT_BALANCE_NON_NEGATIVE",
    condition=F.col("balance") < 0,
    description="Account balance must not be negative"
)

display([result_account_balance])

In [0]:
# Prueba Integridad de Transacciones
df_transactions = spark.table(
    "banco.banking_analytics.silver_transactions"
)

invalid_account_references = (
    df_transactions.alias("t")
    .join(
        df_accounts.select("account_id").alias("a"),
        F.col("t.account_id") == F.col("a.account_id"),
        "left_anti"
    )
)

result_transaction_account_fk = run_quality_test(
    df=invalid_account_references,
    table_name="silver_transactions",
    test_name="ACCOUNT_REFERENCE_VALID",
    condition=F.lit(True),
    description="Every transaction must reference an existing account"
)

display([result_transaction_account_fk])

In [0]:
# Prueba Monto Positivo
result_transaction_amount = run_quality_test(
    df=df_transactions,
    table_name="silver_transactions",
    test_name="TRANSACTION_AMOUNT_POSITIVE",
    condition=F.col("amount") <= 0,
    description="Transaction amount must be greater than zero"
)

display([result_transaction_amount])

In [0]:
# Prueba Fechas de Transaccion
result_transaction_date = run_quality_test(
    df=df_transactions,
    table_name="silver_transactions",
    test_name="TRANSACTION_DATE_NOT_NULL",
    condition=F.col("transaction_date").isNull(),
    description="Transaction date must not be null"
)

display([result_transaction_date])

In [0]:
# Prueba de valores nulos
df_dim_customer = spark.table(
    "banco.banking_analytics.dim_customer"
)

result_customer_key_not_null = run_quality_test(
    df=df_dim_customer,
    table_name="dim_customer",
    test_name="CUSTOMER_KEY_NOT_NULL",
    condition=F.col("customer_key").isNull(),
    description="Customer surrogate key must not be null"
)

display([result_customer_key_not_null])

In [0]:
# Prueba de Claves duplicadas
duplicate_customer_keys = (
    df_dim_customer
    .groupBy("customer_key")
    .count()
    .filter(F.col("count") > 1)
)

result_customer_key_unique = run_quality_test(
    df=duplicate_customer_keys,
    table_name="dim_customer",
    test_name="CUSTOMER_KEY_UNIQUE",
    condition=F.col("count") > 1,
    description="Customer surrogate key must be unique"
)

display([result_customer_key_unique])

In [0]:
# Prueba Integridad Referencial entre Dimensiones
df_dim_account = spark.table(
    "banco.banking_analytics.dim_account"
)

invalid_dim_account_customers = (
    df_dim_account.alias("a")
    .join(
        df_dim_customer.select("customer_key").alias("c"),
        F.col("a.customer_key") == F.col("c.customer_key"),
        "left_anti"
    )
)

result_dim_account_customer_fk = run_quality_test(
    df=invalid_dim_account_customers,
    table_name="dim_account",
    test_name="CUSTOMER_KEY_REFERENCE_VALID",
    condition=F.lit(True),
    description="Every account must reference an existing customer key"
)

display([result_dim_account_customer_fk])

In [0]:
# Prueba integridad Referencial de Tablas Hecho
df_fact_transactions = spark.table(
    "banco.banking_analytics.fact_transactions"
)

invalid_fact_accounts = (
    df_fact_transactions.alias("f")
    .join(
        df_dim_account.select("account_key").alias("a"),
        F.col("f.account_key") == F.col("a.account_key"),
        "left_anti"
    )
)

result_fact_account_fk = run_quality_test(
    df=invalid_fact_accounts,
    table_name="fact_transactions",
    test_name="ACCOUNT_KEY_REFERENCE_VALID",
    condition=F.lit(True),
    description="Every fact transaction must reference an existing account key"
)

display([result_fact_account_fk])

In [0]:
# Prueba Claves foraneas de la tabla de Hechos
df_dim_branch = spark.table(
    "banco.banking_analytics.dim_branch"
)

df_dim_date = spark.table(
    "banco.banking_analytics.dim_date"
)

def find_orphan_keys(fact_df, dimension_df, fact_key, dimension_key):
    return (
        fact_df.alias("f")
        .join(
            dimension_df.select(dimension_key).alias("d"),
            F.col(f"f.{fact_key}") == F.col(f"d.{dimension_key}"),
            "left_anti"
        )
    )

fk_tests = [
    (
        "CUSTOMER_KEY_REFERENCE_VALID",
        "customer_key",
        df_dim_customer,
        "customer_key"
    ),
    (
        "BRANCH_KEY_REFERENCE_VALID",
        "branch_key",
        df_dim_branch,
        "branch_key"
    ),
    (
        "DATE_KEY_REFERENCE_VALID",
        "date_key",
        df_dim_date,
        "date_key"
    )
]

fk_results = []

for test_name, fact_key, dimension_df, dimension_key in fk_tests:

    orphan_df = find_orphan_keys(
        df_fact_transactions,
        dimension_df,
        fact_key,
        dimension_key
    )

    result = run_quality_test(
        df=orphan_df,
        table_name="fact_transactions",
        test_name=test_name,
        condition=F.lit(True),
        description=f"Every fact must reference a valid {dimension_key}"
    )

    fk_results.append(result)

display(fk_results)

In [0]:
# Prueba de Unicidad de Tabla de Hecho
duplicate_transactions = (
    df_fact_transactions
    .groupBy("transaction_id")
    .count()
    .filter(F.col("count") > 1)
)

result_transaction_id_unique = run_quality_test(
    df=duplicate_transactions,
    table_name="fact_transactions",
    test_name="TRANSACTION_ID_UNIQUE",
    condition=F.col("count") > 1,
    description="Each transaction must have exactly one fact record"
)

display([result_transaction_id_unique])

In [0]:
# Prueba consistencia de Montos
silver_reconciliation = (
    df_transactions
    .groupBy("currency")
    .agg(
        F.count("*").alias("silver_count"),
        F.sum("amount").alias("silver_amount")
    )
)

gold_reconciliation = (
    df_fact_transactions
    .groupBy("currency")
    .agg(
        F.count("*").alias("gold_count"),
        F.sum("amount").alias("gold_amount")
    )
)

reconciliation = (
    silver_reconciliation.alias("s")
    .join(
        gold_reconciliation.alias("g"),
        F.col("s.currency") == F.col("g.currency"),
        "full"
    )
    .select(
        F.coalesce(F.col("s.currency"), F.col("g.currency")).alias("currency"),
        "silver_count",
        "gold_count",
        "silver_amount",
        "gold_amount"
    )
    .withColumn(
        "count_difference",
        F.coalesce(F.col("silver_count"), F.lit(0)) -
        F.coalesce(F.col("gold_count"), F.lit(0))
    )
    .withColumn(
        "amount_difference",
        F.coalesce(F.col("silver_amount"), F.lit(0)) -
        F.coalesce(F.col("gold_amount"), F.lit(0))
    )
    .withColumn(
        "status",
        F.when(
            (F.col("count_difference") == 0) &
            (F.col("amount_difference") == 0),
            "PASS"
        ).otherwise("FAIL")
    )
)

display(reconciliation)

In [0]:
# Evaluando un esenario Fail
missing_transactions = (
    df_transactions.alias("s")
    .join(
        df_fact_transactions.select("transaction_id").alias("g"),
        F.col("s.transaction_id") == F.col("g.transaction_id"),
        "left_anti"
    )
)

display(
    missing_transactions.select(
        "transaction_id",
        "account_id",
        "transaction_date",
        "amount",
        "currency",
        "transaction_type",
        "channel",
        "status"
    )
)

In [0]:
# Vrificar que existan la cuenta y las fechas
late_transaction = missing_transactions.select(
    "transaction_id",
    "account_id",
    "transaction_date",
    "amount",
    "currency",
    "transaction_type",
    "channel",
    "status"
)

late_account_check = (
    late_transaction.alias("t")
    .join(
        df_dim_account.alias("a"),
        F.col("t.account_id") == F.col("a.account_id"),
        "left"
    )
    .join(
        df_dim_date.alias("d"),
        F.to_date(F.col("t.transaction_date")) == F.col("d.full_date"),
        "left"
    )
    .select(
        "t.transaction_id",
        "t.account_id",
        "a.account_key",
        "a.customer_key",
        "a.branch_key",
        "d.date_key",
        "t.transaction_date",
        "t.amount",
        "t.currency",
        "t.transaction_type",
        "t.channel",
        "t.status"
    )
)

display(late_account_check)

In [0]:
# Verificando la dimension Date
display(
    df_dim_date
    .agg(
        F.min("full_date").alias("minimum_date"),
        F.max("full_date").alias("maximum_date"),
        F.count("*").alias("total_dates")
    )
)

In [0]:
# Generar fechas hasta fin de año
from pyspark.sql import functions as F

new_dates = (
    spark.range(1)
    .select(
        F.explode(
            F.sequence(
                F.to_date(F.lit("2026-11-01")),
                F.to_date(F.lit("2026-12-31")),
                F.expr("INTERVAL 1 DAY")
            )
        ).alias("full_date")
    )
    .select(
        F.date_format("full_date", "yyyyMMdd").cast("int").alias("date_key"),
        "full_date",
        F.dayofmonth("full_date").alias("day"),
        F.month("full_date").alias("month"),
        F.date_format("full_date", "MMMM").alias("month_name"),
        F.quarter("full_date").alias("quarter"),
        F.year("full_date").alias("year"),
        F.dayofweek("full_date").alias("day_of_week"),
        F.date_format("full_date", "EEEE").alias("day_name"),
        F.weekofyear("full_date").alias("week_of_year"),
        F.dayofweek("full_date").isin([1, 7]).alias("is_weekend")
    )
)

new_dates.printSchema()
display(new_dates.orderBy("full_date"))

In [0]:
# Insertar las nuevas fechas a dim date
from delta.tables import DeltaTable

target_dim_date = DeltaTable.forName(
    spark,
    "banco.banking_analytics.dim_date"
)

(
    target_dim_date.alias("target")
    .merge(
        new_dates.alias("source"),
        "target.date_key = source.date_key"
    )
    .whenNotMatchedInsertAll()
    .execute()
)

# Actualizar el DataFrame para consultar la dimensión vigente
df_dim_date = spark.table("banco.banking_analytics.dim_date")

display(
    df_dim_date.agg(
        F.min("full_date").alias("minimum_date"),
        F.max("full_date").alias("maximum_date"),
        F.count("*").alias("total_dates")
    )
)

In [0]:
df_dim_date.printSchema()

In [0]:
late_account_check = (
    late_transaction.alias("t")
    .join(
        df_dim_account.alias("a"),
        F.col("t.account_id") == F.col("a.account_id"),
        "left"
    )
    .join(
        df_dim_date.alias("d"),
        F.to_date(F.col("t.transaction_date")) == F.col("d.full_date"),
        "left"
    )
    .select(
        "t.transaction_id",
        "t.account_id",
        "a.account_key",
        "a.customer_key",
        "a.branch_key",
        "d.date_key",
        "t.transaction_date",
        "t.amount",
        "t.currency",
        "t.transaction_type",
        "t.channel",
        "t.status"
    )
)

display(late_account_check)

In [0]:
FACT_TRANSACTIONS = "banco.banking_analytics.fact_transactions"

late_fact_row = late_account_check.select(
    "transaction_id",
    "customer_key",
    "account_key",
    "branch_key",
    "date_key",
    "transaction_date",
    "amount",
    "currency",
    "transaction_type",
    "channel",
    "status",
    F.lit(1).cast("long").alias("transaction_count")
)

# MERGE idempotente por identificador de transacción
target_fact = DeltaTable.forName(spark, FACT_TRANSACTIONS)

(
    target_fact.alias("target")
    .merge(
        late_fact_row.alias("source"),
        "target.transaction_id = source.transaction_id"
    )
    .whenNotMatchedInsertAll()
    .execute()
)

# Actualizar la referencia a la tabla Gold
df_fact_transactions = spark.table(FACT_TRANSACTIONS)

# Verificar la transacción incorporada
display(
    df_fact_transactions
    .filter(F.col("transaction_id") == "TRX_LATE_000001")
)

In [0]:
# Validar consistencia
silver_reconciliation = (
    df_transactions
    .groupBy("currency")
    .agg(
        F.count("*").alias("silver_count"),
        F.sum("amount").alias("silver_amount")
    )
)

gold_reconciliation = (
    df_fact_transactions
    .groupBy("currency")
    .agg(
        F.count("*").alias("gold_count"),
        F.sum("amount").alias("gold_amount")
    )
)

reconciliation = (
    silver_reconciliation.alias("s")
    .join(
        gold_reconciliation.alias("g"),
        F.col("s.currency") == F.col("g.currency"),
        "full"
    )
    .select(
        F.coalesce(F.col("s.currency"), F.col("g.currency")).alias("currency"),
        "silver_count",
        "gold_count",
        "silver_amount",
        "gold_amount",
        (F.coalesce(F.col("s.silver_count"), F.lit(0)) -
         F.coalesce(F.col("g.gold_count"), F.lit(0))).alias("count_difference"),
        (F.coalesce(F.col("s.silver_amount"), F.lit(0)) -
         F.coalesce(F.col("g.gold_amount"), F.lit(0))).alias("amount_difference")
    )
    .withColumn(
        "status",
        F.when(
            (F.col("count_difference") == 0) &
            (F.col("amount_difference") == 0),
            "PASS"
        ).otherwise("FAIL")
    )
)

display(reconciliation.orderBy("currency"))

In [0]:
# Comprobar registros y ultima actualización
gold_analytics_tables = [
    "banco.banking_analytics.agg_monthly_transactions",
    "banco.banking_analytics.agg_transaction_status",
    "banco.banking_analytics.agg_transaction_type",
    "banco.banking_analytics.agg_channel_performance",
    "banco.banking_analytics.agg_customer_multichannel"
]

for table_name in gold_analytics_tables:
    print(f"\nTabla: {table_name}")
    print(f"Registros: {spark.table(table_name).count()}")

In [0]:
# REvisar agregación mensual
df_agg_monthly = spark.table(
    "banco.banking_analytics.agg_monthly_transactions"
)

display(
    df_agg_monthly
    .orderBy(F.desc("year"), F.desc("month"), "currency")
    .limit(10)
)

In [0]:
# Recalcular Noviembre
november_2026 = (
    df_fact_transactions
    .filter(
        (F.year("transaction_date") == 2026) &
        (F.month("transaction_date") == 11) &
        (F.col("status") == "COMPLETED")
    )
    .groupBy(
        F.year("transaction_date").alias("year"),
        F.month("transaction_date").alias("month"),
        "currency"
    )
    .agg(
        F.count("*").alias("total_transactions"),
        F.sum("amount").alias("total_amount"),
        F.avg("amount").alias("average_amount"),
        F.countDistinct("customer_key").alias("active_customers")
    )
    .withColumn(
        "year_month",
        F.concat(
            F.col("year"),
            F.lit("-"),
            F.lpad(F.col("month"), 2, "0")
        )
    )
    .withColumn(
        "month_name",
        F.date_format(F.make_date("year", "month", F.lit(1)), "MMMM")
    )
    .select(
        "year",
        "month",
        "year_month",
        "month_name",
        "currency",
        "total_transactions",
        "total_amount",
        "average_amount",
        "active_customers"
    )
)

display(november_2026.orderBy("currency"))

In [0]:
from delta.tables import DeltaTable

MONTHLY_TABLE = "banco.banking_analytics.agg_monthly_transactions"

target_monthly = DeltaTable.forName(spark, MONTHLY_TABLE)

(
    target_monthly.alias("target")
    .merge(
        november_2026.alias("source"),
        """
        target.year = source.year
        AND target.month = source.month
        AND target.currency = source.currency
        """
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

df_agg_monthly = spark.table(MONTHLY_TABLE)

display(
    df_agg_monthly
    .filter(
        (F.col("year") == 2026) &
        (F.col("month") == 11)
    )
    .orderBy("currency")
)

In [0]:
# Revisión de consistenica mensual
monthly_expected = (
    df_fact_transactions
    .filter(F.col("status") == "COMPLETED")
    .groupBy(
        F.year("transaction_date").alias("year"),
        F.month("transaction_date").alias("month"),
        "currency"
    )
    .agg(
        F.count("*").alias("expected_transactions"),
        F.sum("amount").alias("expected_amount"),
        F.avg("amount").alias("expected_average"),
        F.countDistinct("customer_key").alias("expected_active_customers")
    )
)

monthly_actual = spark.table(
    "banco.banking_analytics.agg_monthly_transactions"
)

monthly_comparison = (
    monthly_expected.alias("e")
    .join(
        monthly_actual.alias("a"),
        ["year", "month", "currency"],
        "full"
    )
    .select(
        "year",
        "month",
        "currency",
        "expected_transactions",
        "total_transactions",
        "expected_amount",
        "total_amount",
        "expected_active_customers",
        "active_customers"
    )
    .withColumn(
        "status",
        F.when(
            (F.col("expected_transactions") == F.col("total_transactions")) &
            (F.col("expected_amount") == F.col("total_amount")) &
            (F.col("expected_active_customers") == F.col("active_customers")),
            "PASS"
        ).otherwise("FAIL")
    )
)

display(
    monthly_comparison
    .filter(F.col("status") == "FAIL")
)

In [0]:
display(
    spark.table("banco.banking_analytics.agg_transaction_status")
    .orderBy("status", "currency")
)

In [0]:
status_expected = (
    df_fact_transactions
    .groupBy("currency", "status")
    .agg(
        F.count("*").alias("total_transactions"),
        F.sum("amount").alias("total_amount"),
        F.avg("amount").alias("average_amount"),
        F.countDistinct("customer_key").alias("active_customers"),
        F.countDistinct("account_key").alias("active_accounts"),
        F.countDistinct("branch_key").alias("active_branches")
    )
)

currency_totals = (
    df_fact_transactions
    .groupBy("currency")
    .agg(
        F.count("*").alias("currency_total_transactions")
    )
)

status_expected = (
    status_expected
    .join(currency_totals, "currency")
    .withColumn(
        "status_share_pct",
        F.round(
            F.col("total_transactions") /
            F.col("currency_total_transactions") * 100,
            2
        )
    )
)

display(status_expected.orderBy("currency", "status"))

In [0]:
STATUS_TABLE = "banco.banking_analytics.agg_transaction_status"

target_status = DeltaTable.forName(spark, STATUS_TABLE)

(
    target_status.alias("target")
    .merge(
        status_expected.alias("source"),
        """
        target.currency = source.currency
        AND target.status = source.status
        """
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

df_agg_status = spark.table(STATUS_TABLE)

display(
    df_agg_status.orderBy("currency", "status")
)

In [0]:
status_comparison = (
    status_expected.alias("e")
    .join(
        df_agg_status.alias("a"),
        ["currency", "status"],
        "full"
    )
    .select(
        "currency",
        "status",
        (F.col("e.total_transactions") == F.col("a.total_transactions")).alias("count_match"),
        (F.col("e.total_amount") == F.col("a.total_amount")).alias("amount_match"),
        (F.abs(F.col("e.average_amount") - F.col("a.average_amount")) < F.lit(0.000001)).alias("average_match"),
        (F.col("e.active_customers") == F.col("a.active_customers")).alias("customers_match"),
        (F.col("e.active_accounts") == F.col("a.active_accounts")).alias("accounts_match"),
        (F.col("e.active_branches") == F.col("a.active_branches")).alias("branches_match"),
        (F.col("e.currency_total_transactions") == F.col("a.currency_total_transactions")).alias("currency_total_match"),
        (F.col("e.status_share_pct") == F.col("a.status_share_pct")).alias("share_match")
    )
    .withColumn(
        "status",
        F.when(
            F.col("count_match") &
            F.col("amount_match") &
            F.col("average_match") &
            F.col("customers_match") &
            F.col("accounts_match") &
            F.col("branches_match") &
            F.col("currency_total_match") &
            F.col("share_match"),
            "PASS"
        ).otherwise("FAIL")
    )
)

display(status_comparison.orderBy("currency", "status"))

In [0]:
TYPE_TABLE = "banco.banking_analytics.agg_transaction_type"

df_agg_type = spark.table(TYPE_TABLE)

print("Esquema de la tabla:")
df_agg_type.printSchema()

print("Cantidad de registros:")
print(df_agg_type.count())

display(df_agg_type.orderBy("currency", "transaction_type"))

In [0]:
type_expected = (
    df_fact_transactions
    .groupBy("transaction_type", "currency")
    .agg(
        F.count("*").alias("total_transactions"),
        F.sum("amount").alias("total_amount"),
        F.avg("amount").alias("average_amount"),
        F.min("amount").alias("minimum_amount"),
        F.max("amount").alias("maximum_amount"),
        F.countDistinct("customer_key").alias("active_customers"),
        F.countDistinct("account_key").alias("active_accounts")
    )
)

display(type_expected.orderBy("transaction_type", "currency"))

In [0]:
type_expected = (
    df_fact_transactions
    .filter(F.col("status") == "COMPLETED")
    .groupBy("transaction_type", "currency")
    .agg(
        F.count("*").alias("total_transactions"),
        F.sum("amount").alias("total_amount"),
        F.avg("amount").alias("average_amount"),
        F.min("amount").alias("minimum_amount"),
        F.max("amount").alias("maximum_amount"),
        F.countDistinct("customer_key").alias("active_customers"),
        F.countDistinct("account_key").alias("active_accounts")
    )
)

display(type_expected.orderBy("transaction_type", "currency"))

In [0]:
TYPE_TABLE = "banco.banking_analytics.agg_transaction_type"

target_type = DeltaTable.forName(spark, TYPE_TABLE)

(
    target_type.alias("target")
    .merge(
        type_expected.alias("source"),
        """
        target.transaction_type = source.transaction_type
        AND target.currency = source.currency
        """
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

df_agg_type = spark.table(TYPE_TABLE)

display(df_agg_type.orderBy("transaction_type", "currency"))

In [0]:
type_comparison = (
    type_expected.alias("e")
    .join(
        df_agg_type.alias("a"),
        ["transaction_type", "currency"],
        "full"
    )
    .select(
        "transaction_type",
        "currency",
        (F.col("e.total_transactions") == F.col("a.total_transactions")).alias("count_match"),
        (F.col("e.total_amount") == F.col("a.total_amount")).alias("amount_match"),
        (F.abs(F.col("e.average_amount") - F.col("a.average_amount")) < F.lit(0.000001)).alias("average_match"),
        (F.col("e.minimum_amount") == F.col("a.minimum_amount")).alias("minimum_match"),
        (F.col("e.maximum_amount") == F.col("a.maximum_amount")).alias("maximum_match"),
        (F.col("e.active_customers") == F.col("a.active_customers")).alias("customers_match"),
        (F.col("e.active_accounts") == F.col("a.active_accounts")).alias("accounts_match")
    )
    .withColumn(
        "validation_status",
        F.when(
            F.col("count_match") &
            F.col("amount_match") &
            F.col("average_match") &
            F.col("minimum_match") &
            F.col("maximum_match") &
            F.col("customers_match") &
            F.col("accounts_match"),
            "PASS"
        ).otherwise("FAIL")
    )
)

display(type_comparison.orderBy("transaction_type", "currency"))

In [0]:
CHANNEL_TABLE = "banco.banking_analytics.agg_channel_performance"

df_agg_channel = spark.table(CHANNEL_TABLE)

print("Esquema de la tabla:")
df_agg_channel.printSchema()

print("Cantidad de registros:")
print(df_agg_channel.count())

display(df_agg_channel.orderBy("currency", "channel"))

In [0]:
channel_expected = (
    df_fact_transactions
    .groupBy("channel", "currency")
    .agg(
        F.count("*").alias("total_transactions"),
        F.sum("amount").alias("total_amount"),
        F.avg("amount").alias("average_amount"),
        F.countDistinct("customer_key").alias("active_customers"),
        F.countDistinct("account_key").alias("active_accounts"),
        F.countDistinct("branch_key").alias("active_branches")
    )
)

display(channel_expected.orderBy("currency", "channel"))

In [0]:
channel_expected = (
    df_fact_transactions
    .filter(F.col("status") == "COMPLETED")
    .groupBy("channel", "currency")
    .agg(
        F.count("*").alias("total_transactions"),
        F.sum("amount").alias("total_amount"),
        F.avg("amount").alias("average_amount"),
        F.countDistinct("customer_key").alias("active_customers"),
        F.countDistinct("account_key").alias("active_accounts"),
        F.countDistinct("branch_key").alias("active_branches")
    )
)

display(channel_expected.orderBy("currency", "channel"))

In [0]:
CHANNEL_TABLE = "banco.banking_analytics.agg_channel_performance"

target_channel = DeltaTable.forName(spark, CHANNEL_TABLE)

(
    target_channel.alias("target")
    .merge(
        channel_expected.alias("source"),
        """
        target.channel = source.channel
        AND target.currency = source.currency
        """
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

df_agg_channel = spark.table(CHANNEL_TABLE)

display(df_agg_channel.orderBy("currency", "channel"))

In [0]:
channel_comparison = (
    channel_expected.alias("e")
    .join(
        df_agg_channel.alias("a"),
        ["channel", "currency"],
        "full"
    )
    .select(
        "channel",
        "currency",
        (F.col("e.total_transactions") == F.col("a.total_transactions")).alias("count_match"),
        (F.col("e.total_amount") == F.col("a.total_amount")).alias("amount_match"),
        (F.abs(F.col("e.average_amount") - F.col("a.average_amount")) < F.lit(0.000001)).alias("average_match"),
        (F.col("e.active_customers") == F.col("a.active_customers")).alias("customers_match"),
        (F.col("e.active_accounts") == F.col("a.active_accounts")).alias("accounts_match"),
        (F.col("e.active_branches") == F.col("a.active_branches")).alias("branches_match")
    )
    .withColumn(
        "validation_status",
        F.when(
            F.col("count_match") &
            F.col("amount_match") &
            F.col("average_match") &
            F.col("customers_match") &
            F.col("accounts_match") &
            F.col("branches_match"),
            "PASS"
        ).otherwise("FAIL")
    )
)

display(channel_comparison.orderBy("currency", "channel"))